In [2]:
import pandas as pd
import seaborn as srn
import statistics  as sts
pd.options.display.float_format = '{:,.0f}'.format


In [3]:
dataset = pd.read_csv("../DRE_Estatais_Bruto/dfp_dre_2014.csv", encoding='utf-8', sep=";")
dataset.shape
dataset = dataset.rename(columns={
    'CNPJ_CIA':      'cnpj',
    'DT_REFER':      'data_referencia',
    'VERSAO':        'versao',
    'DENOM_CIA':     'empresa',
    'CD_CVM':        'codigo_cvm',
    'ESCALA_MOEDA':  'escala',
    'ORDEM_EXERC':   'exercicio',
    'DT_INI_EXERC':  'data_inicio_exercicio',
    'DT_FIM_EXERC':  'data_fim_exercicio',
    'CD_CONTA':      'codigo_conta',
    'DS_CONTA':      'descricao_conta',
    'VL_CONTA': 'valor',
    'ST_CONTA_FIXA': 'conta_fixa',
})

dataset['exercicio'] = dataset['exercicio'].map({'ÚLTIMO': 'atual', 'PENÚLTIMO': 'anterior'})
dataset = dataset[dataset.exercicio == 'atual']
dataset = dataset.drop(columns=['GRUPO_DFP','MOEDA'])
dataset = dataset.drop(columns=['exercicio', 'versao'])

dataset.head()


,cnpj,data_referencia,empresa,codigo_cvm,escala,data_inicio_exercicio,data_fim_exercicio,codigo_conta,descricao_conta,valor,conta_fixa
1,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-01-01,2014-12-31,3.01,Receitas da Intermediação Financeira,"137,778,601",S
3,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-01-01,2014-12-31,3.01.01,Receita de Juros,"137,778,601",S
5,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-01-01,2014-12-31,3.02,Despesas da Intermediação Financeira,"-105,909,418",S
7,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-01-01,2014-12-31,3.02.01,Despesas de Juros,"-91,124,202",S
9,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-01-01,2014-12-31,3.02.02,Despesa com Provisão para Perdas em Empréstimo...,"-14,789,246",S


In [4]:
empresas_selecionadas = {
    'BCO BRASIL S.A.':                                ('Financeiro', 'Estatal'),
    'ITAU UNIBANCO HOLDING S.A.':                     ('Financeiro', 'Privado'),
    'BCO BRADESCO S.A.':                              ('Financeiro', 'Privado'),
    'BCO SANTANDER (BRASIL) S.A.':                    ('Financeiro', 'Privado'),
    'BCO BTG PACTUAL S.A.':                           ('Financeiro', 'Privado'),
    'BCO ABC BRASIL S.A.':                            ('Financeiro', 'Privado'),
    'BCO PAN S.A.':                                   ('Financeiro', 'Privado'),
    'CIA ENERGETICA DE MINAS GERAIS - CEMIG':         ('Energia', 'Estatal'),
    'CIA PARANAENSE DE ENERGIA - COPEL':              ('Energia', 'Estatal'),
    'CENTRAIS ELET BRAS S.A. - ELETROBRAS':           ('Energia', 'Estatal'),
    'ENGIE BRASIL ENERGIA S.A.':                      ('Energia', 'Privado'),
    'EQUATORIAL ENERGIA S.A.':                        ('Energia', 'Privado'),
    'CPFL ENERGIA S.A.':                              ('Energia', 'Privado'),
    'ENERGISA S.A.':                                  ('Energia', 'Privado'),
    'ALUPAR INVESTIMENTO S/A':                        ('Energia', 'Privado'),
    'NEOENERGIA S.A.':                                ('Energia', 'Privado'),
    'PETROLEO BRASILEIRO S.A. PETROBRAS':             ('Petroleo', 'Estatal'),
    'PRIO S.A.':                                      ('Petroleo', 'Privado'),
}

dataset = dataset[dataset['empresa'].isin(empresas_selecionadas.keys())].copy()

dataset['setor'] = dataset.empresa.map(lambda x: empresas_selecionadas[x][0])  # type: ignore
dataset['tipo'] = dataset.empresa.map(lambda x: empresas_selecionadas[x][1]) # type: ignore

dataset.head()




,cnpj,data_referencia,empresa,codigo_cvm,escala,data_inicio_exercicio,data_fim_exercicio,codigo_conta,descricao_conta,valor,conta_fixa,setor,tipo
1,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-01-01,2014-12-31,3.01,Receitas da Intermediação Financeira,"137,778,601",S,Financeiro,Estatal
3,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-01-01,2014-12-31,3.01.01,Receita de Juros,"137,778,601",S,Financeiro,Estatal
5,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-01-01,2014-12-31,3.02,Despesas da Intermediação Financeira,"-105,909,418",S,Financeiro,Estatal
7,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-01-01,2014-12-31,3.02.01,Despesas de Juros,"-91,124,202",S,Financeiro,Estatal
9,00.000.000/0001-91,2014-12-31,BCO BRASIL S.A.,1023,MIL,2014-01-01,2014-12-31,3.02.02,Despesa com Provisão para Perdas em Empréstimo...,"-14,789,246",S,Financeiro,Estatal


In [13]:
dre = pd.read_csv("../DRE_Tratado/dfp_dre_completo.csv", sep=";", encoding="utf-8")

# Ver todas as contas únicas com descrição
contas = (
    dre[['codigo_conta', 'descricao_conta', 'setor']]
    .drop_duplicates()
    .sort_values(['setor', 'codigo_conta'])
)

print("\n=== PETROLEO ===")
print(contas[contas['setor'] == 'Petroleo'].to_string())



=== PETROLEO ===
     codigo_conta                                                            descricao_conta     setor
310          3.01                                     Receita de Venda de Bens e/ou Serviços  Petroleo
311          3.02                                      Custo dos Bens e/ou Serviços Vendidos  Petroleo
312          3.03                                                            Resultado Bruto  Petroleo
313          3.04                                             Despesas/Receitas Operacionais  Petroleo
314       3.04.01                                                        Despesas com Vendas  Petroleo
315       3.04.02                                          Despesas Gerais e Administrativas  Petroleo
316    3.04.02.01                                          Despesas com Geologia e Geofísica  Petroleo
317    3.04.02.02                                                       Despesas com Pessoal  Petroleo
318    3.04.02.03                                      